# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../firef/E Commerce Dataset.xlsx"),
    Path("firef/E Commerce Dataset.xlsx"),
    Path("/Users/firef/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()
df.info()

读取文件：..\firef\E Commerce Dataset.xlsx
数据形状：5630 行，20 列
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus        

### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [3]:
df.info()

# 答案：
# 1.以为客户在平台上的用户画像信息及历史行为汇总数据
# 2.CustomerID
# 3.Churn

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [5]:
missing_report = None
# 计算每列缺失值的数量
missing_count = df.isna().sum()

# 计算每列缺失值的比例
missing_ratio = df.isna().mean()

# 将数量和比例合并到一个新的 DataFrame 中，并命名为 missing_report
missing_report = pd.DataFrame({
    '缺失数量': missing_count,
    '缺失比例': missing_ratio
})

# 打印查看结果
print(missing_report)

                             缺失数量  缺失比例
CustomerID                      0  0.00
Churn                           0  0.00
Tenure                        264  0.05
PreferredLoginDevice            0  0.00
CityTier                        0  0.00
WarehouseToHome               251  0.04
PreferredPaymentMode            0  0.00
Gender                          0  0.00
HourSpendOnApp                255  0.05
NumberOfDeviceRegistered        0  0.00
PreferedOrderCat                0  0.00
SatisfactionScore               0  0.00
MaritalStatus                   0  0.00
NumberOfAddress                 0  0.00
Complain                        0  0.00
OrderAmountHikeFromlastYear   265  0.05
CouponUsed                    256  0.05
OrderCount                    258  0.05
DaySinceLastOrder             307  0.05
CashbackAmount                  0  0.00


### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [6]:
# 统计完全重复的行数（即所有列的值都一模一样的行）
duplicate_rows = df.duplicated().sum()

# 统计 CustomerID 列中重复出现的次数（即同一个客户有多条记录）
duplicate_customer_ids = df.duplicated(subset=['CustomerID']).sum()

# 打印查看结果
print("完全重复行数：", duplicate_rows)
print("CustomerID 重复数量：", duplicate_customer_ids)
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

完全重复行数： 0
CustomerID 重复数量： 0


---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [8]:
# TODO：循环填充每列的中位数
for col in numeric_missing_cols:
    df[col].fillna(df[col].median(), inplace=True)
    

# TODO：输出上述字段剩余的缺失值数量
print(df[numeric_missing_cols].isna().sum())


Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
dtype: int64


### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [9]:
中位数的数学定义;第50分位数。稳定抗极值（稳健性）
因此在需要极值的情况或者数据的极值无意义时不能使用中位数填充（局限性：信息丢失和极值失效）

不适合使用中位数填充的情况有

1分类/名义变量：中位数对类别型数据无数学意义（如“颜色”“地区”），只能用众数填充。
2缺失机制为MNAR（非随机缺失）：数据缺失与缺失值本身相关（如高收入人群故意不填收入），此时任何常数填充都会产生系统性偏差，必须定性分析或选用模型插补。
3缺失率过高（>30%~40%）：单值填充会人为制造“尖峰”分布，严重扭曲真实分布形态，此时应放弃填充或使用多重插补。
4数据存在明显分组差异（辛普森悖论）：若总体中位数与各组中位数差异巨大（如不同部门薪资），直接填充会掩盖组间差异，必须按分组分别计算中位数填充。
5后续使用距离/方差敏感型算法：KNN、聚类、PCA等依赖距离的算法，中位数填充会缩小样本间距离，导致“抱团”而丧失区分度；线性回归中会低估标准误，增大假阳性风险。
6时间序列数据：直接用全局中位数会破坏时间趋势和自相关性，应优先考虑线性插值或前向/后向填充。
7离散型计数数据：若中位数为非整数（如7.5），直接填充会产生无意义值，应保留整数逻辑（如众数或四舍五入后的中位数，且需业务验证）。

若必须填充，先按分组计算中位数；若方差需保留，可改用随机抽样填充（从非缺失值中随机抽取）或多重插补。


SyntaxError: invalid character '。' (U+3002) (801706541.py, line 1)

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [10]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64


### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [13]:
# TODO：完成类别标准化
# 将 Phone 统一替换为 Mobile Phone
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({"Phone": "Mobile Phone"})

# 将 COD 和 CC 统一替换为全称
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})
# 将 Mobile 统一替换为 Mobile Phone
df["PreferedOrderCat"] = df["PreferedOrderCat"].replace({"Mobile": "Mobile Phone"})

# TODO：重新检查类别频数
# 假设你的 category_cols 列表包含了这三个字段，如果没有可以手动指定
category_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [14]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [15]:
# TODO：调用函数检查三个字段
display(iqr_outlier_summary(df["WarehouseToHome"]))
display(iqr_outlier_summary(df["OrderCount"]))
display(iqr_outlier_summary(df["CashbackAmount"]))

Q1         9.00
Q3        20.00
下限        -7.50
上限        36.50
候选异常值数量    2.00
dtype: float64

Q1          1.00
Q3          3.00
下限         -2.00
上限          6.00
候选异常值数量   703.00
dtype: float64

Q1        145.77
Q3        196.39
下限         69.84
上限        272.33
候选异常值数量   438.00
dtype: float64

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [16]:
# TODO：完成业务规则检查
rules = {
    "使用时长小于 0": (df["HourSpendOnApp"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}
# 将结果转换为 Series 方便查看
pd.Series(rules)

使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
dtype: int64

---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [17]:
# TODO：完成验收
# 1. 确认指定数值字段不再有缺失
assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"

# 2. 确认类别同义值已统一
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

# 3. 确认输出目录存在（补充逻辑）
import os
output_dir = "./output"  # 请根据实际路径修改
os.makedirs(output_dir, exist_ok=True)  # 如果目录不存在则自动创建

print("✅ 数据清洗验收通过。")

✅ 数据清洗验收通过。


### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [18]:
# TODO：导出清洗后的数据
output_path = os.path.join(output_dir, "cleaned_data.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"📁 数据已成功导出至: {output_path}")
from pathlib import Path

OUTPUT_PATH = Path("../output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"✅ 已导出：{OUTPUT_PATH.resolve()}")

📁 数据已成功导出至: ./output\cleaned_data.csv


PermissionError: [Errno 13] Permission denied: '..\\output\\ecommerce_customer_cleaned.csv'

---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？